# Dialectal Bias in Hate Speech Detection Systems

By: Sérgio Costa

A step is skipped if the output file already exists (set `FORCE = True` to redo everything).

In [50]:
import os, subprocess, sys

FORCE = False  # True = run every step again even if the output already exists

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")


def run(script, skip_if=None):
    """Run one script of the pipeline and show the output as it is produced."""
    if skip_if and not FORCE and all(os.path.exists(p) for p in skip_if):
        print(f"[skipped] {script}: output already exists ({', '.join(skip_if)})")
        return
    print(f"\n>>> {script}\n")
    env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
    process = subprocess.Popen([sys.executable, script], env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"{script} failed (exit code {process.returncode})")

### Step 1 - Get the datasets
Davidson et al. (2017) tweets (ground truth), the Twitter part of HateXplain and the HateDay dataset

In [51]:
# In case you want to download the HateDay dataset, don't forget to login on hugging face here:
# from huggingface_hub import login
# login()

In [52]:
run("src/download_dataset/get_davidson_dataset.py", skip_if=["data/raw/davidson_dataset.csv"])
run("src/download_dataset/get_hatexplain_dataset.py", skip_if=["data/raw/hatexplain_twitter.csv"])
run("src/download_dataset/get_hateday_dataset.py", skip_if=["data/raw/hateday_us.csv"])

[skipped] src/download_dataset/get_davidson_dataset.py: output already exists (data/raw/davidson_dataset.csv)
[skipped] src/download_dataset/get_hatexplain_dataset.py: output already exists (data/raw/hatexplain_twitter.csv)
[skipped] src/download_dataset/get_hateday_dataset.py: output already exists (data/raw/hateday_us.csv)


### Step 2 - Score the dialect of every tweet
The model from Blodgett et al. (2016) is used here to give the probability of a tweet being AAE-aligned. The classes are `aae_aligned` (for a threshold ≥ 0.6), `mainstream` (for a threshold ≤ 0.4), `ambiguous` or `unscored`.

In [53]:
run("src/score_dialect/score_dialect_blodgett.py", skip_if=["data/processed/davidson_with_dialect.csv", "data/processed/hatexplain_with_dialect.csv", "data/processed/hateday_with_dialect.csv"])

[skipped] src/score_dialect/score_dialect_blodgett.py: output already exists (data/processed/davidson_with_dialect.csv, data/processed/hatexplain_with_dialect.csv, data/processed/hateday_with_dialect.csv)


### Step 3 - Classify with the two hate speech models
1. `cardiffnlp/twitter-roberta-base-hate`
2. `unitary/unbiased-toxic-roberta`

In [54]:
run("src/classification/predict_hate_classifiers.py")


>>> src/classification/predict_hate_classifiers.py


------------------------------  CLASSIFIERS: DAVIDSON  ------------------------------
Found 24776 saved predictions.

Model 1 (cardiffnlp/twitter-roberta-base-hate): 24776 tweets already done, 0 to run.

Model 2 (unitary/unbiased-toxic-roberta): 24776 tweets already done, 0 to run.

Saved 24776 predictions to data\processed\davidson_predictions.csv

------------------------------  CLASSIFIERS: HATEXPLAIN  ------------------------------
Found 8777 saved predictions.

Model 1 (cardiffnlp/twitter-roberta-base-hate): 8777 tweets already done, 0 to run.

Model 2 (unitary/unbiased-toxic-roberta): 8777 tweets already done, 0 to run.

Saved 8777 predictions to data\processed\hatexplain_predictions.csv

------------------------------  CLASSIFIERS: HATEDAY  ------------------------------
Found 26970 saved predictions.

Model 1 (cardiffnlp/twitter-roberta-base-hate): 26970 tweets already done, 0 to run.

Model 2 (unitary/unbiased-toxic-roberta

### Step 4 - Evaluation
#### Step 4.1 - Davidson Dataset
##### Step 4.1.1 Baseline evaluation

In [55]:
run("src/evaluation/baseline_evaluation.py")


>>> src/evaluation/baseline_evaluation.py



MODEL 1: twitter-roberta-base-hate

 ------------------------------  1. Confusion matrix per group  ------------------------------

aae_aligned (n=4689)
                    not flagged     flagged         
real not-hate       TN = 1123       FP = 3384       
real hate           FN = 75         TP = 107        

mainstream (n=13211)
                    not flagged     flagged         
real not-hate       TN = 8083       FP = 4136       
real hate           FN = 844        TP = 148        

 ------------------------------  2. Metrics per group ------------------------------

--- aae_aligned (n=4689) ---
              precision    recall  f1-score   support

    not_hate       0.94      0.25      0.39      4507
        hate       0.03      0.59      0.06       182

    accuracy                           0.26      4689
   macro avg       0.48      0.42      0.23      4689
weighted avg       0.90      0.26      0.38      4689


--- mainstream (n

##### Step 4.1.2 Hate-flag rate by dialect group, within each human label

In [56]:
run("src/evaluation/dialect_bias_by_label.py")


>>> src/evaluation/dialect_bias_by_label.py



MODEL 1: twitter-roberta-base-hate
--- % of tweets the model flags as hate, by human label ---
dialect_group       aae_aligned  mainstream
label                                      
hate_speech                58.8        14.9
neither                     9.5         1.6
offensive_language         76.5        48.0

--- number of tweets ---
dialect_group       aae_aligned  mainstream
label                                      
hate_speech                 182         992
neither                      95        3724
offensive_language         4412        8495


MODEL 2: unbiased-toxic-roberta
--- % of tweets the model flags as hate, by human label ---
dialect_group       aae_aligned  mainstream
label                                      
hate_speech                84.1        85.7
neither                    25.3        16.1
offensive_language         79.7        85.5

--- number of tweets ---
dialect_group       aae_aligned  mainstream
label  

##### Step 4.1.3 Dialect bias independent of the default threshold: AUC, score gap and FPR at matched thresholds

In [57]:
run("src/evaluation/threshold_free_dialect_bias.py")


>>> src/evaluation/threshold_free_dialect_bias.py

17900 tweets: 4689 AAE-aligned, 13211 mainstream

Model 1 (twitter-roberta-base-hate)

--- 1. AUC per group (positive vs negative class of this model, Davidson labels) ---
AAE-aligned AUC: 0.392    mainstream AUC: 0.497    gap: -0.104  95% CI [-0.145, -0.059]

--- 2. Score difference between groups in the negtive tweets ---
P(AAE score > mainstream score) = 0.761   95% CI [0.753, 0.768]   (n = 4507 / 12219)
Median score: AAE-aligned 0.775   mainstream 0.205

--- 3. FPR at a fixed overall flag rate (same threshold for both groups) ---
flag rate   threshold   FPR AAE   FPR main  gap       
0.05        0.9277      0.107     0.032     +0.075    
0.10        0.9011      0.207     0.066     +0.141    
0.20        0.8326      0.395     0.139     +0.256    
0.30        0.7262      0.559     0.220     +0.338    
0.50        0.3632      0.822     0.400     +0.422    

--- 4. AAE-aligned FPR when the mainstream FPR is fixed ---
main FPR    thres

##### Step 4.1.4 Lexical controls (does vocabulary explain the dialect gap)?
**Step 4.1.4.1 Keyword control to women-directed profanity**

Tests whether the gap between AAE-aligned and mainstream tweets comes from a small set of women-directed profane terms (and their spelling variants) that are much more common in AAE-aligned tweets. Tweets are split by whether they contain a term, and the two groups are compared inside each split. The script reports:
- the flag-rate gap at the default threshold;
- the threshold-free score comparison, P(AAE score > mainstream score);
- how strongly each group’s flag rate reacts to the terms;
- the gap after reweighting mainstream to the AAE share of tweets with a term.

If the gap persists without the terms, these words do not explain it. For model 2 (toxicity), flags on offensive tweets are correct detections, so false positives are read from the neither tweets only.

In [58]:
run("src/evaluation/keyword_control_dialect_bias.py")


>>> src/evaluation/keyword_control_dialect_bias.py


--- 1. How common are the terms? ---
offensive language   AAE-aligned 0.910   mainstream 0.772
hate speech          AAE-aligned 0.648   mainstream 0.158
neither              AAE-aligned 0.063   mainstream 0.030

--- 2. Flag rate gap at the default threshold, split by the terms ---
subset                            model    n AAE n main     AAE    main     gap   95% CI
offensive, WITH a term            model 1  4013   6555   0.830   0.606  +0.225   [+0.209, +0.241]
offensive, WITH a term            model 2  4013   6555   0.816   0.884  -0.068   [-0.082, -0.054]
offensive, WITHOUT any term       model 1   399   1940   0.108   0.054  +0.054   [+0.023, +0.085]
offensive, WITHOUT any term       model 2   399   1940   0.604   0.758  -0.154   [-0.207, -0.102]
offensive, all                    model 1  4412   8495   0.765   0.480  +0.285   [+0.269, +0.302]
offensive, all                    model 2  4412   8495   0.797   0.855  -0.059   [-0.

**Step 4.1.4.2 Vocabulary control (how much of the dialect gap a bag-of-words model reproduces)**

Generalises the keyword control without a hand-made word list. Among the not-hate tweets, a bag-of-words model is trained to predict each classifier’s output from which words a tweet contains. The predictions are cross-validated, so each tweet is predicted by a model that did not see it.

In [59]:
run("src/evaluation/vocabulary_control_dialect_bias.py")


>>> src/evaluation/vocabulary_control_dialect_bias.py


Model 1 (twitter-roberta-base-hate)

--- 1a. Words with the highest flag rate ---
word                tweets  flag rate  AAE share
she                    611      0.848      0.535
bitches               1831      0.845      0.442
yo                     360      0.806      0.808
her                    617      0.796      0.397
these                  786      0.790      0.756
she's                  108      0.778      0.028
niggas                 444      0.770      0.905
dick                   203      0.768      0.453
fat                    122      0.754      0.385
bad                    340      0.753      0.282
hoes                  1352      0.752      0.706
girl                   246      0.752      0.313
ain't                  566      0.746      0.853
ugly                   131      0.740      0.351
bitch                 5113      0.726      0.320
gone                   129      0.721      0.845
em                     109  

##### Step 4.1.5 Robustness to the dialect group definition

In [60]:
run("src/evaluation/group_definition_robustness.py")


>>> src/evaluation/group_definition_robustness.py

--- 1. Label mix of the groups under each definition (% of the group) ---
definition                            group              n    hate  offensive  neither
1. baseline (0.6 / 0.4)               AAE-aligned     4689     3.9       94.1      2.0
1. baseline (0.6 / 0.4)               mainstream     13211     7.5       64.3     28.2
2. argmax (AAE vs White)              AAE-aligned    13533     3.8       91.6      4.6
2. argmax (AAE vs White)              mainstream      6669     7.6       52.0     40.3
3. profanity removed before scoring   AAE-aligned     2387     4.1       92.2      3.7
3. profanity removed before scoring   mainstream     17068     6.3       71.8     21.9
4a. stricter thresholds 0.7 / 0.3     AAE-aligned     2157     4.1       94.7      1.2
4a. stricter thresholds 0.7 / 0.3     mainstream      9518     8.8       56.1     35.1
4b. stricter thresholds 0.8 / 0.2     AAE-aligned      580     5.0       94.1      0.9
4b. 

#### Step 4.2 Evaluate on the HateXplain (Twitter) Dataset

In [61]:
run("src/evaluation/evaluate_hatexplain.py")


>>> src/evaluation/evaluate_hatexplain.py

--- 1. Label mix of the groups under each definition (% of the group) ---
definition                            group              n    hate  offensive  neither
1. baseline (0.6 / 0.4)               AAE-aligned      522     6.1       43.5     50.4
1. baseline (0.6 / 0.4)               mainstream      7230     8.1       23.5     68.5
2. argmax (AAE vs White)              AAE-aligned     1975     7.9       39.0     53.0
2. argmax (AAE vs White)              mainstream      5166     6.8       20.7     72.5
3. profanity removed before scoring   AAE-aligned      275     6.2       38.2     55.6
3. profanity removed before scoring   mainstream      7650     8.0       24.7     67.3
4a. stricter thresholds 0.7 / 0.3     AAE-aligned      216     3.7       44.4     51.9
4a. stricter thresholds 0.7 / 0.3     mainstream      6248     7.6       22.3     70.1
4b. stricter thresholds 0.8 / 0.2     AAE-aligned       46     4.3       52.2     43.5
4b. stricter

#### Step 4.3 Evaluate on the HateDay Dataset

In [62]:
run("src/evaluation/evaluate_hateday.py")


>>> src/evaluation/evaluate_hateday.py

--- 1. Label mix of the groups under each definition (% of the group) ---
definition                            group              n    hate  offensive  neither
1. baseline (0.6 / 0.4)               AAE-aligned      397     0.5       27.5     72.0
1. baseline (0.6 / 0.4)               mainstream     25110     0.4        5.4     94.2
2. argmax (AAE vs White)              AAE-aligned     2937     0.2       16.9     82.9
2. argmax (AAE vs White)              mainstream     17141     0.5        5.7     93.8
3. profanity removed before scoring   AAE-aligned      350     0.3       19.4     80.3
3. profanity removed before scoring   mainstream     25234     0.4        5.9     93.8
4a. stricter thresholds 0.7 / 0.3     AAE-aligned      168     0.0       22.0     78.0
4a. stricter thresholds 0.7 / 0.3     mainstream     22973     0.4        5.0     94.6
4b. stricter thresholds 0.8 / 0.2     AAE-aligned       55     0.0       14.5     85.5
4b. stricter th

In [63]:
run("src/evaluation/analyze_hateday_controls.py")


>>> src/evaluation/analyze_hateday_controls.py

23938 'neither' tweets: 286 AAE-aligned, 23652 mainstream

=== 1. Profanity and length in each group ===
AAE-aligned  n=   286   with profanity: 0.049   median words: 4
mainstream   n= 23652   with profanity: 0.006   median words: 10
words       AAE-aligned   mainstream   (% of each group)
1-3                39.5         15.7
4-6                19.2         15.0
7-10               16.1         16.6
11+                16.4         49.9
Vocabulary 'basic': 1703 words (min_df 20)
Vocabulary 'extended': 3329 words (min_df 10)

Model 1 (twitter-roberta-base-hate)
Flagged tweets among 'neither': AAE-aligned 9 of 286, mainstream 183 of 23652

--- 2. With and without profanity ---
subset                     n AAE n main flagged AAE    AAE   main     gap   95% CI            P(A>M)  95% CI
WITH profanity                14    141   too few tweets in one group
WITHOUT profanity            272  23511           8  0.029  0.008  +0.022   [+0.003, +0.04

#### Step 4.4 Summary

In [64]:
run("src/evaluation/summarize_three_corpora.py")


>>> src/evaluation/summarize_three_corpora.py


Tweets labelled 'neither', groups defined by: baseline

corpus      model                       n AAE  n main flagged  FPR AAE FPR main     gap   95% CI            P(A>M)  95% CI           r@10%  r@20%
Davidson    twitter-roberta-base-hate      95    3724       9    0.095    0.016  +0.078   [+0.025, +0.142]  0.674  [0.622, 0.727]   2.42   2.21
Davidson    unbiased-toxic-roberta         95    3724      24    0.253    0.161  +0.091   [+0.002, +0.180]  0.656  [0.603, 0.697]   1.79   1.53
HateXplain  twitter-roberta-base-hate     263    4950      97    0.369    0.121  +0.248   [+0.188, +0.308]  0.749  [0.720, 0.777]   3.42   2.53
HateXplain  unbiased-toxic-roberta        263    4950     201    0.764    0.460  +0.304   [+0.249, +0.357]  0.700  [0.663, 0.737]   3.00   2.59
HateDay     twitter-roberta-base-hate     286   23652       9    0.031    0.008  +0.024   [+0.006, +0.045]  0.733  [0.708, 0.755]   2.66   2.41
HateDay     unbiased-toxic-ro